In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torch.nn.utils as utils
import torchvision.utils as vutils
import numpy as np
from torch.utils.data import DataLoader, Dataset
import cv2
import os

**Data**

In [ ]:
image_dir = 'dataset/train/'

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, .5, .5]),
])


class Flower102(Dataset):
    def __init__(self, transform=None):
        image_paths = [list(map(lambda name: class_name + '/' + name, os.listdir(image_dir + class_name))) for class_name in os.listdir(image_dir)]
        image_paths = sum(image_paths, [])
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = os.path.join(image_dir, self.image_paths[idx])
        img = Image.open(img_path).convert('RGB')
        return self.transform(img)

dataset = Flower102(transform=transform)

data_loader = DataLoader(dataset, batch_size=128, shuffle=True, pin_memory=True, drop_last=True)

**Architechture**

In [ ]:
def ConvBlock(in_channels, out_channels, kernel_size=3, stride=1, padding=1, dropout=.1):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.GELU(),
        nn.BatchNorm2d(out_channels),
        nn.Conv2d(out_channels, out_channels, 1),
        nn.GELU(),
        nn.Dropout2d(dropout),
        nn.BatchNorm2d(out_channels)
    )

In [ ]:
class Encoder(nn.Module):
    def __init__(self, w_dim, hidden_dim):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.c1 = ConvBlock(3, 64 * w_dim)
        self.c2 = ConvBlock(64 * w_dim, 128 * w_dim)
        self.c3 = ConvBlock(128 * w_dim, 2 * hidden_dim)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
    
    def forward(self, x):
        x = self.c1(x)
        x = self.pool(x)
        x = self.c2(x)
        x = self.pool(x)
        x = self.c3(x)
        x = self.avgpool(x)
        mu = x[-1, :hidden_dim, ...]
        logvar = x[-1, hidden_dim:, ...]
        return mu, logvar

In [ ]:
class Decoder(nn.Module):
    def __init__(self, w_dim, hidden_dim):
        super().__init__()

        self.w_dim = w_dim
        self.hidden_dim = hidden_dim

        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear')
        self.setsize = nn.Upsample(size=16)

        self.c1 = ConvBlock(hidden_dim, 128 * w_dim)
        self.c2 = ConvBlock(128 * w_dim, 64 * w_dim)
        self.c3 = ConvBlock(64 * w_dim, 3)
        self.out = nn.Sigmoid()
    

    def reparametrization(self, mu, logvar):
        eps = torch.randn(self.hidden_dim)
        return mu + eps * torch.exp(logvar)

    def forward(self, x):
        x = x.view(-1, self.hidden_dim, 1, 1)
        x = self.setsize(x)
        x = self.c1(x)
        x = self.upsample(x)
        x = self.c2(x)
        x = self.upsample(x)
        x = self.c3(x)
        x = self.out(x)
        return x


In [ ]:
class AE(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc
        self.dec = dec
        self.reparam = self.dec.reparametrization
    
    def generate(self, z):
        return self.dec(z)
    
    def forward(self, x):
        pass

**Models definition**

In [ ]:
# Параметры
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

w_dim = 1
hidden_dim = 128

encoder = Encoder(w_dim, hidden_dim)
decoder = Decoder(w_dim, hidden_dim)

ae = AE(encoder, decoder).to(device)

optimizer = optim.AdamW(ae.parameters(), lr=3e-4)


**Image Generation / Presentation**

In [ ]:
def generate_images(ae, n_images=4):
    with torch.no_grad():
        noise = torch.randn(n_images, hidden_dim, device=device, requires_grad=False)
        images = ae.generate(noise).detach().cpu()
    return images

In [ ]:
def show_images(ae, title, n_images=4):
    images = generate_images(ae, n_images)
    # make_grid сам соберет картинки в одну строку (nrow=len(images))
    # normalize=True сам переведет из диапазона сети в [0, 1]
    grid = vutils.make_grid(images.detach().cpu(), nrow=len(images), padding=2, normalize=True)

    plt.figure(figsize=(len(images) * 2, 3))
    plt.axis("off")
    plt.title(f"epoch №{title}")

    # make_grid возвращает (C, H, W), поэтому всё равно нужно permute для imshow
    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.show()

In [ ]:
show_images(ae, "test")

**Training Loop**

In [ ]:
criterion = nn.BCELoss()
n_epochs = 20


def beta(epoch, max_epochs, max_weight=1.0):
    return max_weight * min(1.0, epoch / (max_epochs * 0.5))

def kl_loss(logvar, mu):
    return -.5 * torch.mean(1 + logvar - mu * mu - logvar.exp())

def rec_loss(imgs, rec_imgs):
    return criterion(rec_imgs, imgs)

def loss_fn(imgs, rec_imgs, mu, logvar, epoch):
    return rec_loss(imgs, rec_imgs) + beta(epoch, n_epochs) * kl_loss(logvar, mu)

In [ ]:
for epoch in range(n_epochs):
    train_loss = 0

    for batch_id, data in enumerate(data_loader):
        imgs = data.to(device, non_blocking=True)

        mu, logvar = ae.enc(imgs)
        z = ae.reparam(mu, logvar)
        rec_imgs = ae.dec(z)
        
        loss = loss_fn(imgs, rec_imgs, mu, logvar, epoch)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()

        ae.zero_grad()

    # Callback
    train_loss /= data_loader.__len__()
    print(f"Epoch \t[{epoch}/{n_epochs}], Loss: {abs(train_loss):.3f}")
    show_images(ae, title=epoch, n_images=5)


 **Вывод сгенерированных картинок**

In [ ]:
show_images(netG, title='test', n_images=15)

In [ ]:
import seaborn as sns

sns.lineplot(discriminator_losses[60:])

In [ ]:
show_images(g_ema.get_model(), title='test', n_images=15)

In [ ]:
save_models(netG=netG, nameG=2, netD=netD, nameD=2)
assert False

In [ ]:
load_models(netG=netG, nameG=497, netD=netD, nameD=497)
load_models(netG=g_ema.get_model(), nameG='gEMA_497', netD=netD, nameD=497)

In [ ]:
show_images(g_ema.get_model(), title='test', n_images=15)

In [ ]:
show_images(g_ema.get_model(), title='test', n_images=15)